In [1]:
#!ollama pull llama3.2

In [2]:
#!pip install langchain-ollama

In [18]:
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side

def sablon_olustur():
    wb = Workbook()
    ws = wb.active
    ws.title = "Denetim Verileri"

    # --- TASARIM AYARLARI ---
    bold_font = Font(bold=True, color="FFFFFF")
    header_fill = PatternFill(start_color="4F81BD", end_color="4F81BD", fill_type="solid")
    center_align = Alignment(horizontal="center", vertical="center")
    
    # Sütun Genişlikleri
    ws.column_dimensions['A'].width = 50  # Açıklama sütunu geniş olsun
    ws.column_dimensions['B'].width = 20  # TP
    ws.column_dimensions['C'].width = 20  # YP

    # --- BAŞLIKLAR ---
    ws['A1'] = "BANKA ADI:"
    ws['A2'] = "DÖNEM:"
    
    # Tablo Başlıkları (Satır 4)
    headers = ["KALEMLER", "TP (Türk Parası)", "YP (Yabancı Para)"]
    ws.append([]) # 3. satırı boş geç
    ws.append(headers) # 4. satıra başlıkları yaz

    # Başlık Hücrelerini Boya
    for cell in ws[4]:
        cell.font = bold_font
        cell.fill = header_fill
        cell.alignment = center_align

    # --- SATIR ETİKETLERİ (A Sütunu) ---
    kalemler = [
        "Nakit ve Nakit Benzerleri",                      # Satır 5
        "G.U.D Farkı K/Z Yansıtılan Finansal Varlıklar",  # Satır 6
        "G.U.D Farkı Diğer Kapsamlı Gelire Yansıtılan",   # Satır 7
        "İtfa Edilmiş Maliyeti ile Ölçülen",              # Satır 8
        "Beklenen Zarar Karşılıkları (-)",                # Satır 9
        "Ortaklık Yatırımları",                           # Satır 10
        "Maddi Duran Varlıklar (Net)",                    # Satır 11
        "Maddi Olmayan Duran Varlıklar (Net)",            # Satır 12
        "Diğer Aktifler",                                 # Satır 13
        "VARLIKLAR TOPLAMI"                               # Satır 14
    ]

    for i, kalem in enumerate(kalemler, start=5):
        hucre = ws[f'A{i}']
        hucre.value = kalem
        hucre.font = Font(bold=True)

    # --- KAYDETME ---
    dosya_adi = "sablon.xlsx" 
    wb.save(dosya_adi)
    print(f"Harika! '{dosya_adi}' dosyası başarıyla oluşturuldu.")

if __name__ == "__main__":
    sablon_olustur()

Harika! 'sablon.xlsx' dosyası başarıyla oluşturuldu.


In [19]:
import os
from typing import Optional
from langchain_ollama import ChatOllama
from langchain_community.document_loaders import PyPDFLoader
from pydantic import BaseModel, Field
from openpyxl import load_workbook

In [20]:
from pydantic import BaseModel, Field
from typing import Optional

class DenetimVerileri(BaseModel):
    # --- GENEL BİLGİLER ---
    banka_adi: str = Field(description="Raporun en üstünde yer alan bankanın tam ticari unvanı.")
    donem: str = Field(description="Raporun ait olduğu tarih ve dönem bilgisi (Örn: 30 Haziran 2025).")

    # --- 1. NAKİT VE NAKİT BENZERLERİ ---
    nakit_benzerleri_tp: Optional[int] = Field(description="Cari Dönem sütunundaki 'Nakit ve Nakit Benzerleri' satırının TP (Türk Parası) değeri.")
    nakit_benzerleri_yp: Optional[int] = Field(description="Cari Dönem sütunundaki 'Nakit ve Nakit Benzerleri' satırının YP (Yabancı Para) değeri.")

    # --- 2. GERÇEĞE UYGUN DEĞER FARKI K/Z YANSITILAN ---
    fv_kar_zarar_tp: Optional[int] = Field(description="Cari Dönem sütunundaki 'Gerçeğe Uygun Değer Farkı Kâr Zarara Yansıtılan Finansal Varlıklar' satırının TP değeri.")
    fv_kar_zarar_yp: Optional[int] = Field(description="Cari Dönem sütunundaki 'Gerçeğe Uygun Değer Farkı Kâr Zarara Yansıtılan Finansal Varlıklar' satırının YP değeri.")

    # --- 3. GERÇEĞE UYGUN DEĞER FARKI DİĞER KAPSAMLI GELİRE YANSITILAN ---
    fv_diger_kapsamli_tp: Optional[int] = Field(description="Cari Dönem sütunundaki 'Gerçeğe Uygun Değer Farkı Diğer Kapsamlı Gelire Yansıtılan Finansal Varlıklar' satırının TP değeri.")
    fv_diger_kapsamli_yp: Optional[int] = Field(description="Cari Dönem sütunundaki 'Gerçeğe Uygun Değer Farkı Diğer Kapsamlı Gelire Yansıtılan Finansal Varlıklar' satırının YP değeri.")

    # --- 4. İTFA EDİLMİŞ MALİYETİ İLE ÖLÇÜLEN ---
    itfa_edilmis_tp: Optional[int] = Field(description="Cari Dönem sütunundaki 'İtfa Edilmiş Maliyeti ile Ölçülen Finansal Varlıklar' satırının TP değeri.")
    itfa_edilmis_yp: Optional[int] = Field(description="Cari Dönem sütunundaki 'İtfa Edilmiş Maliyeti ile Ölçülen Finansal Varlıklar' satırının YP değeri.")

    # --- 5. BEKLENEN ZARAR KARŞILIKLARI (-) ---
    beklenen_zarar_tp: Optional[int] = Field(description="Cari Dönem sütunundaki 'Beklenen Zarar Karşılıkları (-)' satırının TP değeri. (Genellikle İtfa Edilmiş Maliyet veya Krediler altındaki).")
    beklenen_zarar_yp: Optional[int] = Field(description="Cari Dönem sütunundaki 'Beklenen Zarar Karşılıkları (-)' satırının YP değeri.")

    # --- 6. ORTAKLIK YATIRIMLARI ---
    ortaklik_yatirimlari_tp: Optional[int] = Field(description="Cari Dönem sütunundaki 'ORTAKLIK YATIRIMLARI' veya 'İştirakler' ana başlığının TP değeri.")
    ortaklik_yatirimlari_yp: Optional[int] = Field(description="Cari Dönem sütunundaki 'ORTAKLIK YATIRIMLARI' veya 'İştirakler' ana başlığının YP değeri.")

    # --- 7. MADDİ DURAN VARLIKLAR (NET) ---
    maddi_duran_tp: Optional[int] = Field(description="Cari Dönem sütunundaki 'MADDİ DURAN VARLIKLAR (Net)' satırının TP değeri.")
    maddi_duran_yp: Optional[int] = Field(description="Cari Dönem sütunundaki 'MADDİ DURAN VARLIKLAR (Net)' satırının YP değeri.")

    # --- 8. MADDİ OLMAYAN DURAN VARLIKLAR (NET) ---
    maddi_olmayan_tp: Optional[int] = Field(description="Cari Dönem sütunundaki 'MADDİ OLMAYAN DURAN VARLIKLAR (Net)' satırının TP değeri.")
    maddi_olmayan_yp: Optional[int] = Field(description="Cari Dönem sütunundaki 'MADDİ OLMAYAN DURAN VARLIKLAR (Net)' satırının YP değeri.")

    # --- 9. DİĞER AKTİFLER ---
    diger_aktifler_tp: Optional[int] = Field(description="Cari Dönem sütunundaki 'DİĞER AKTİFLER' satırının TP değeri.")
    diger_aktifler_yp: Optional[int] = Field(description="Cari Dönem sütunundaki 'DİĞER AKTİFLER' satırının YP değeri.")

    # --- 10. VARLIKLAR TOPLAMI ---
    varliklar_toplami_tp: Optional[int] = Field(description="Cari Dönem sütunundaki 'VARLIKLAR TOPLAMI' veya 'AKTİF TOPLAMI' satırının TP değeri.")
    varliklar_toplami_yp: Optional[int] = Field(description="Cari Dönem sütunundaki 'VARLIKLAR TOPLAMI' veya 'AKTİF TOPLAMI' satırının YP değeri.")


In [21]:
llm = ChatOllama(model="llama3.2", temperature=0)

# Modeli, yukarıdaki şemayı (JSON) çıktı verecek şekilde ayarlıyoruz.
structured_llm = llm.with_structured_output(DenetimVerileri)

In [22]:
def veriyi_cek_ve_yaz(pdf_yolu, excel_sablon_yolu, cikti_yolu):
    print(f"{pdf_yolu} işleniyor...")

    # --- A. PDF OKUMA ---
    try:
        loader = PyPDFLoader(pdf_yolu)
        pages = loader.load()

        # Şimdilik ilk 10 sayfayı alalım örnek olarak:
        full_text = "".join([p.page_content for p in pages[9:11]])
        
    except Exception as e:
        print(f"PDF okuma hatası: {e}")
        return

    # --- B. AI İLE VERİ ÇIKARMA ---
    print("AI verileri analiz ediyor...")
    prompt = f"""
    Aşağıdaki banka denetim raporu metnini analiz et ve istenen finansal verileri çıkar.
    Eğer veriyi bulamazsan ilgili alan için null/None döndür. Asla tahmin yürütme.
    
    RAPOR METNİ:
    {full_text}
    """
    
    # Yapılandırılmış veriyi alıyoruz
    veri_sonucu = structured_llm.invoke(prompt)
    print(f"Çıkarılan Veriler: {veri_sonucu}")

    print("Excel'e yazılıyor...")
    try:
        wb = load_workbook(excel_sablon_yolu)
        ws = wb.active 

        # Banka Adı ve Dönem (Header)
        ws['B1'] = veri_sonucu.banka_adi
        ws['B2'] = veri_sonucu.donem

        # 1. Nakit ve Nakit Benzerleri
        ws['B5'] = veri_sonucu.nakit_benzerleri_tp
        ws['C5'] = veri_sonucu.nakit_benzerleri_yp

        # 2. Gerçeğe Uygun Değer Farkı K/Z
        ws['B6'] = veri_sonucu.fv_kar_zarar_tp
        ws['C6'] = veri_sonucu.fv_kar_zarar_yp

        # 3. Gerçeğe Uygun Değer Farkı Diğer Kapsamlı
        ws['B7'] = veri_sonucu.fv_diger_kapsamli_tp
        ws['C7'] = veri_sonucu.fv_diger_kapsamli_yp

        # 4. İtfa Edilmiş Maliyet
        ws['B8'] = veri_sonucu.itfa_edilmis_tp
        ws['C8'] = veri_sonucu.itfa_edilmis_yp

        # 5. Beklenen Zarar Karşılıkları
        ws['B9'] = veri_sonucu.beklenen_zarar_tp
        ws['C9'] = veri_sonucu.beklenen_zarar_yp

        # 6. Ortaklık Yatırımları
        ws['B10'] = veri_sonucu.ortaklik_yatirimlari_tp
        ws['C10'] = veri_sonucu.ortaklik_yatirimlari_yp

        # 7. Maddi Duran Varlıklar
        ws['B11'] = veri_sonucu.maddi_duran_tp
        ws['C11'] = veri_sonucu.maddi_duran_yp

        # 8. Maddi Olmayan Duran Varlıklar
        ws['B12'] = veri_sonucu.maddi_olmayan_tp
        ws['C12'] = veri_sonucu.maddi_olmayan_yp

        # 9. Diğer Aktifler
        ws['B13'] = veri_sonucu.diger_aktifler_tp
        ws['C13'] = veri_sonucu.diger_aktifler_yp

        # 10. Varlıklar Toplamı
        ws['B14'] = veri_sonucu.varliklar_toplami_tp
        ws['C14'] = veri_sonucu.varliklar_toplami_yp

        wb.save(cikti_yolu)
        print(f"Başarılı! Veriler {cikti_yolu} dosyasına kaydedildi.")

    except Exception as e:
        print(f"Excel yazma hatası: {e}")
# --- ÇALIŞTIRMA ---
if __name__ == "__main__":
    # Dosya yollarını kendi bilgisayarınıza göre düzenleyin
    pdf_dosyasi = "ornek_banka_raporu.pdf"
    sablon_dosyasi = "sablon.xlsx"
    sonuc_dosyasi = "sonuc_2024.xlsx"

    

In [23]:

if __name__ == "__main__":
    # 1. PDF dosyanızın tam yolu (İsmi neyse onu yazın)
    pdf_dosyasi = "/Users/hussain/Desktop/Project[X]/Aktif Yatırım Bankası Solo Rapor 30.06.2025.pdf" 
    
    # 2. Boş Excel şablonunuzun tam yolu
    sablon_dosyasi = "/Users/hussain/Desktop/Project[X]/sablon.xlsx"
    
    # 3. Sonucun kaydedileceği yer ve isim (Bunu kod oluşturacak)
    sonuc_dosyasi = "/Users/hussain/Desktop/Project[X]/sonuc_2025.xlsx"

    # FONKSİYONU ÇAĞIRMA (Başındaki # işaretini kaldırdık)
    # Artık kod çalıştığında bu satır devreye girecek.
    veriyi_cek_ve_yaz(pdf_dosyasi, sablon_dosyasi, sonuc_dosyasi)

/Users/hussain/Desktop/Project[X]/Aktif Yatırım Bankası Solo Rapor 30.06.2025.pdf işleniyor...
AI verileri analiz ediyor...
Çıkarılan Veriler: banka_adi='Aktif Yatırımlar Bankası' donem='2025' nakit_benzerleri_tp=1000000 nakit_benzerleri_yp=500000 fv_kar_zarar_tp=2000000 fv_kar_zarar_yp=1000000 fv_diger_kapsamli_tp=3000000 fv_diger_kapsamli_yp=1500000 itfa_edilmis_tp=5000000 itfa_edilmis_yp=2500000 beklenen_zarar_tp=10000000 beklenen_zarar_yp=5000000 ortaklik_yatirimlari_tp=20000000 ortaklik_yatirimlari_yp=10000000 maddi_duran_tp=15000000 maddi_duran_yp=7500000 maddi_olmayan_tp=5000000 maddi_olmayan_yp=2500000 diger_aktifler_tp=10000000 diger_aktifler_yp=5000000 varliklar_toplami_tp=200000000 varliklar_toplami_yp=100000000
Excel'e yazılıyor...
Başarılı! Veriler /Users/hussain/Desktop/Project[X]/sonuc_2025.xlsx dosyasına kaydedildi.
